# Classifying movement in fiction
Optional practice after the Class 7 exit note. You may ask AI about code or errors. Read the passages and choose and justify a setting yourself.

In [ ]:
#@title Setup (run only)
import json
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
from sklearn.linear_model import LogisticRegression


url = "https://nlp.lang.osaka-u.ac.jp/dh/b/materials/class7-consensus-v1.zip"
archive, _ = urlretrieve(url)
with ZipFile(archive) as bundle:
    passages = [json.loads(line) for line in bundle.open("passages.jsonl")]
    ballots = [json.loads(line) for line in bundle.open("llm-labels.jsonl")]
    reader = json.load(bundle.open("reader-labels.json"))
    with np.load(bundle.open("embeddings.npz")) as data:
        vectors = dict(zip(data["ids"], data["vectors"].astype("float32")))
items = {p["id"]: p for p in passages}
train = [b for b in ballots if items[b["id"]]["split"] == "train" and b["label"]]
validation = [b for b in ballots if items[b["id"]]["split"] == "dev" and b["label"]]
valid = [b for b in validation if items[b["id"]].get("sampling") != "negation-cue"]
targeted = [b for b in validation if items[b["id"]].get("sampling") == "negation-cue"]
train_x = [vectors[b["id"]] for b in train]
train_y = [b["label"] for b in train]

cohorts = {'random': 'Random sample', 'rare-9b': 'Rare historical labels'}

def show_passage(i):
    p = items[i]
    print(p["author"], p["title"], i)
    print(p["previous"])
    print(p["left"] + "【" + p["target"] + "】" + p["right"])
    print(p["next"])

def validation_scores(sample):
    ids = [b["id"] for b in sample]
    predictions = model.predict([vectors[i] for i in ids])
    hits = guesses = reference = agreements = 0
    for row, prediction in zip(sample, predictions):
        gold = row["label"]
        agreements += prediction == gold
        reference += gold == "CT-"
        guesses += prediction == "CT-"
        hits += gold == prediction == "CT-"
    def ratio(numerator, denominator):
        if denominator:
            return f"{numerator} / {denominator}"
        return '— (zero denominator)'

    print('Agreement', ratio(agreements, len(sample)))
    print('CT− recall', ratio(hits, reference))
    print('CT− precision', ratio(hits, guesses))
    print('False CT− predictions', guesses - hits)
    for label in ("CT+", "CT-", "Uu", "NM"):
        actual = [pred for row, pred in zip(sample, predictions) if row["label"] == label]
        if actual:
            print(label, 'recall', ratio(actual.count(label), len(actual)))

def reveal_reading(i):
    print('Reader', reader[i]["label"], reader[i]["evidence"])
    print('Classifier prediction', model.predict([vectors[i]])[0])

def reader_scores():
    first_disagreement = None
    for cohort, name in cohorts.items():
        ids = [i for i, r in reader.items() if r["cohort"] == cohort and r["label"] != "defer"]
        agreements = 0
        for i, pred in zip(ids, model.predict([vectors[i] for i in ids])):
            if reader[i]["label"] == pred:
                agreements += 1
            elif first_disagreement is None:
                first_disagreement = i
        print(name, agreements, "/", len(ids))
    return first_disagreement

case = next(b for b in train if b["agreement"] == 2)
reading_ids = [b["id"] for b in targeted if b["label"] == "CT-"]
print('Ready')

## 1. Read a disputed passage
Use CT+ when the narrator presents vertical movement as occurring, CT− when it fails to occur, Uu when its occurrence is open, and NM for another sense. Choose a label and supporting phrase before revealing the votes.

In [ ]:
show_passage(case["id"])

### Reveal the three votes

In [ ]:
for name, vote in case["votes"].items():
    if vote:
        print(name, vote["label"], vote["evidence"])
print('Majority vote', case["label"])

## 2. Read four validation passages
Find words around each verb that tell you whether the movement took place.

In [ ]:
for passage_id in reading_ids:
    show_passage(passage_id)

Run with standard weights first. Then change None to "balanced" below and rerun fitting and scoring. Compare the CT− passages recovered with the extra CT− predictions.

In [ ]:
weights = None

In [ ]:
model = LogisticRegression(class_weight=weights, max_iter=2000)
model.fit(train_x, train_y)
print('Original validation')
validation_scores(valid)
print('Negation-cue validation')
validation_scores(targeted)

## 3. Compare with the reader
Record your setting and reason first. All 20 works were held out. One of 12 random passages was deferred because the marked span was faulty. Eight cases chosen for rare historical Qwen labels are scored separately.

In [ ]:
example = reader_scores()
show_passage(example)

Which label would you give the marked verb? Identify a phrase supporting your reading before running the reveal cell.

In [ ]:
reveal_reading(example)